## End-to-End Verification

Smoke test notebook — verifies that the full pipeline ran correctly across all layers.
Does not modify any table. Read-only checks.

### Checks

**Bronze**
- All tables have rows for all 3 cities

**Silver**
- Row count > 0 for every table and city
- `is_valid` rate > 90% on stops
- No null `trip_id` in stop_times

**Gold**
- All marts have rows for all 3 cities
- No null `metric_value` in city_comparison
- `total_routes` > 0 for every city in kpi_network

### Success Criteria
All checks return expected counts with no nulls and no missing cities.

## Bronze

In [0]:
SELECT 
    'bronze.stops' AS table_name, city, 
    COUNT(*) AS row_count,
    CASE WHEN COUNT(*) > 0 THEN 'PASS' ELSE 'FAIL' END AS check_result
FROM gtfs_bronze.stops
GROUP BY city
UNION ALL
SELECT 
    'bronze.routes' AS table_name, 
    city, 
    COUNT(*) AS row_count,
    CASE WHEN COUNT(*) > 0 THEN 'PASS' ELSE 'FAIL' END AS check_result
FROM gtfs_bronze.routes
GROUP BY city
UNION ALL
SELECT 
    'bronze.trips' AS table_name, 
    city, 
    COUNT(*) AS row_count,
    CASE WHEN COUNT(*) > 0 THEN 'PASS' ELSE 'FAIL' END AS check_result
FROM gtfs_bronze.trips
GROUP BY city
UNION ALL
SELECT 
    'bronze.stop_times' AS table_name, 
    city, 
    COUNT(*) AS row_count,
    CASE WHEN COUNT(*) > 0 THEN 'PASS' ELSE 'FAIL' END AS check_result
FROM gtfs_bronze.stop_times
GROUP BY city
UNION ALL
SELECT 
    'bronze.calendar_dates' AS table_name, 
    city, 
    COUNT(*) AS row_count,
    CASE WHEN COUNT(*) > 0 THEN 'PASS' ELSE 'FAIL' END AS check_result
FROM gtfs_bronze.calendar_dates
GROUP BY city
UNION ALL
SELECT 
    'bronze.calendar' AS table_name, 
    city, 
    COUNT(*) AS row_count,
    CASE WHEN COUNT(*) > 0 THEN 'PASS' ELSE 'FAIL' END AS check_result
FROM gtfs_bronze.calendar
GROUP BY city

## Silver

### Count check

In [0]:
SELECT 
    'silver.calendar' AS table_name, 
    city, 
    COUNT(*) AS row_count,
    CASE WHEN COUNT(*) > 0 THEN 'PASS' ELSE 'FAIL' END AS check_result
FROM gtfs_silver.calendar
GROUP BY city
UNION ALL
SELECT 
    'silver.calendar_dates' AS table_name, 
    city, 
    COUNT(*) AS row_count,
    CASE WHEN COUNT(*) > 0 THEN 'PASS' ELSE 'FAIL' END AS check_result
FROM gtfs_silver.calendar_dates
GROUP BY city
UNION ALL
SELECT 
    'silver.routes' AS table_name, 
    city, 
    COUNT(*) AS row_count,
    CASE WHEN COUNT(*) > 0 THEN 'PASS' ELSE 'FAIL' END AS check_result
FROM gtfs_silver.routes
GROUP BY city
UNION ALL
SELECT 
    'silver.stop_times' AS table_name, 
    city, 
    COUNT(*) AS row_count,
    CASE WHEN COUNT(*) > 0 THEN 'PASS' ELSE 'FAIL' END AS check_result
FROM gtfs_silver.stop_times
GROUP BY city
UNION ALL
SELECT 
    'silver.stops' AS table_name, 
    city, 
    COUNT(*) AS row_count,
    CASE WHEN COUNT(*) > 0 THEN 'PASS' ELSE 'FAIL' END AS check_result
FROM gtfs_silver.stops
GROUP BY city
UNION ALL
SELECT 
    'silver.trips' AS table_name, 
    city, 
    COUNT(*) AS row_count,
    CASE WHEN COUNT(*) > 0 THEN 'PASS' ELSE 'FAIL' END AS check_result
FROM gtfs_silver.trips
GROUP BY city

### Quality check

In [0]:
SELECT city,
    COUNT(*) AS total,
    SUM(CASE WHEN is_valid THEN 1 ELSE 0 END) AS valid_count,
    ROUND(100.0 * SUM(CASE WHEN is_valid THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_valid,
    CASE WHEN ROUND(100.0 * SUM(CASE WHEN is_valid THEN 1 ELSE 0 END) / COUNT(*), 1) >= 90 
         THEN 'PASS' ELSE 'FAIL' END AS check_result
FROM gtfs_silver.stops
GROUP BY city

In [0]:
SELECT city,
    SUM(CASE WHEN trip_id IS NULL THEN 1 ELSE 0 END) AS null_trip_id,
    CASE WHEN SUM(CASE WHEN trip_id IS NULL THEN 1 ELSE 0 END) = 0 
         THEN 'PASS' ELSE 'FAIL' END AS check_result
FROM gtfs_silver.stop_times
GROUP BY city

## Gold

### Count check

In [0]:
SELECT 
    'gold.city_comparison' AS table_name, 
    city, 
    COUNT(*) AS row_count,
    CASE WHEN COUNT(*) > 0 THEN 'PASS' ELSE 'FAIL' END AS check_result
FROM gtfs_gold.city_comparison
GROUP BY city
UNION ALL
SELECT
    'gold.hourly_coverage' AS table_name,
    city,
    COUNT(*) AS row_count,
    CASE WHEN COUNT(*) > 0 THEN 'PASS' ELSE 'FAIL' END AS check_result
FROM gtfs_gold.hourly_coverage
GROUP BY city
UNION ALL
SELECT 
    'gold.kpi_network' AS table_name,
    city,
    COUNT(*) AS row_count,
    CASE WHEN COUNT(*) > 0 THEN 'PASS' ELSE 'FAIL' END AS check_result
FROM gtfs_gold.kpi_network
GROUP BY city
UNION ALL
SELECT 
    'gold.od_flows' AS table_name,
    city,
    COUNT(*) AS row_count,
    CASE WHEN COUNT(*) > 0 THEN 'PASS' ELSE 'FAIL' END AS check_result
FROM gtfs_gold.od_flows
GROUP BY city

### Quality check

In [0]:
SELECT 
    city,
    SUM(CASE WHEN total_routes = 0 THEN 1 ELSE 0 END) AS null_total_routes,
    ROUND(100.0 * SUM(CASE WHEN total_routes = 0 THEN 1 ELSE 0 END) / COUNT(*),1) AS pct_null_total_routes,
    CASE WHEN ROUND(100.0 * SUM(CASE WHEN total_routes = 0 THEN 1 ELSE 0 END) / COUNT(*),1) > 0 
         THEN 'FAIL' ELSE 'PASS' END AS check_result
FROM gtfs_gold.kpi_network
GROUP BY city

In [0]:
SELECT city,
    SUM(CASE WHEN metric_value IS NULL THEN 1 ELSE 0 END) AS null_metric_value,
    CASE WHEN SUM(CASE WHEN metric_value IS NULL THEN 1 ELSE 0 END) = 0 
         THEN 'PASS' ELSE 'FAIL' END AS check_result
FROM gtfs_gold.city_comparison
GROUP BY city